# molecule-gpt — a character-level GPT that writes molecules

Trains the from-scratch transformer in this repo on **MOSES** (~1.6M drug-like SMILES),
then scores what it invents with RDKit: **validity**, **uniqueness**, **novelty**.
The last section steers generation toward a target drug-likeness (QED) and shows the shift.

**Before running:** Settings -> Internet **ON**, Accelerator **GPU T4**.
All logic lives in the repo; this notebook only calls it.

In [ ]:
import os
%cd /kaggle/working
if os.path.exists('molecule-gpt'):
    !git -C molecule-gpt pull
else:
    !git clone https://github.com/hi-z-k/molecule-gpt
%cd /kaggle/working/molecule-gpt

In [ ]:
!pip install -q rdkit 2>&1 | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Data

Downloads MOSES, canonicalizes every SMILES with RDKit, drops anything that does not parse or is
longer than 100 characters, and computes QED and logP per molecule so generation can be conditioned
on them. Bin edges are quantiles of the training split only.

In [ ]:
!python scripts/prepare_data.py --out_dir data/processed 2>&1 | tail -15

## 2. Train

Character-level next-token prediction. Nothing about chemistry is hard-coded: ring closures
(`C1...C1`), branch brackets and valences all have to be learned from the strings alone.
Validity is measured on freshly generated samples at every eval step, so you can watch the
grammar being acquired.

In [ ]:
!python -m gpt.train --max_minutes 100 --run_name smiles 2>&1 | tail -25

In [ ]:
from IPython.display import Image, display
display(Image('results/smiles_loss.png'))

## 3. What did it invent?

5,000 unconditional samples, scored against the training set.

In [ ]:
!python -m mol.sample --ckpt checkpoints/smiles/best.pt --n 5000 \
    --out results/uncond.json --grid results/uncond_grid.png 2>&1 | tail -20

In [ ]:
display(Image('results/uncond_grid.png'))

## 4. Steering: ask for a drug-likeness and check it obeys

The prompt carries a QED bin token. If conditioning works, the mean QED of what comes out should
track the bin that was asked for — the model is not just reproducing the training distribution,
it can be aimed at part of it.

In [ ]:
import json, subprocess

rows = []
for b in [0, 3, 6, 9]:
    out = f'results/qed_bin{b}.json'
    subprocess.run(['python', '-m', 'mol.sample', '--ckpt', 'checkpoints/smiles/best.pt',
                    '--n', '1000', '--qed_bin', str(b), '--out', out], check=True)
    m = json.load(open(out))
    qed = float('nan') if m['qed'] is None else m['qed']
    rows.append((b, m['validity'], m['uniqueness'], qed))
    print(f"bin {b}: validity {m['validity']:.3f}  unique {m['uniqueness']:.3f}  mean QED {qed:.3f}")

In [ ]:
import matplotlib.pyplot as plt

bins = [r[0] for r in rows]
qed = [r[3] for r in rows]
plt.figure(figsize=(5, 3.5))
plt.plot(bins, qed, 'o-')
plt.xlabel('requested QED bin (0 = least drug-like, 9 = most)')
plt.ylabel('mean QED of generated molecules')
plt.title('Does the model follow the property it was asked for?')
plt.grid(alpha=.3)
plt.tight_layout()
plt.savefig('results/steering.png', dpi=120)
plt.show()

## 5. The honest limit

These are *candidates*: strings that are valid chemistry and look like the drug-like distribution.
Nothing here shows any of them binds a target or is safe. That needs docking and lab work.
What is demonstrated is that the model learned SMILES grammar and the shape of drug-like chemical
space from next-character prediction alone, and can be pointed at a region of it.

In [ ]:
import shutil, os
out = '/kaggle/working/outputs'
shutil.rmtree(out, ignore_errors=True)
os.makedirs(out)
shutil.copy('checkpoints/smiles/best.pt', out)
shutil.copytree('results', out + '/results')
shutil.make_archive('/kaggle/working/molecule-gpt-outputs', 'zip', out)
print(sorted(os.listdir(out + '/results')))